# Algorithmic Methods of Data Mining and Laboratory

# Lab Session 06 - Principal Component Analysis

In [ ]:
from sklearn import datasets
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
!gdown 1QLoqVEp9KnCbQ5kGToEYS-nVq4xqQ_i_

## Singular Value Decomposition

Performing singular value decomposition on a matrix $\mathbf{A} \in \mathbb{R}^{n \times m}$ with rank $r$ consists of finding $r$ triplets $\{(\sigma_i, \mathbf{u}_i, \mathbf{v}_i)\}_{i = 1}^n$ such that
$$\mathbf{A} = \sum_{i = 1}^r \sigma_i\mathbf{u}_i\mathbf{v}_i^T = \begin{pmatrix}
  \mathbf{u}_1 & \dots & \mathbf{u}_r \\
\end{pmatrix}\begin{pmatrix}
  \sigma_1 & 0 & \dots & 0 \\
  0 & \sigma_2 & \dots & 0 \\
  \vdots & \vdots & \ddots & \vdots \\
  0 & 0 & \dots & \sigma_r \\
\end{pmatrix}\begin{pmatrix}
  \mathbf{v}_1^T \\
  \vdots \\
  \mathbf{v}_r^T \\
\end{pmatrix} = \mathbf{U}_r\mathbf{\Sigma}_r\mathbf{V}_r^T$$

In [ ]:
# Load and visualize the dataset.
x = np.load("Lab_SVD_data.npy")
x

# Assume that each row corresponds to a user and each column represents an item.
n, m = x.shape
print(n,"users")
print(m,"items")

# Perform singular value decomposition using the native command.
U, Sigma, VT = np.linalg.svd(x, full_matrices = False, compute_uv = True) # Computing the full matrices U and V is not necessary.
print("U:",U.shape)
print("Sigma:",Sigma.shape)
print("VT:",VT.shape)

# Take a look at the diagonal matrix storing the singular values.
print(Sigma) # The first three dimensions appear to be the most informative.

While singular value decomposition is often performed on complete matrices, it is also possible to cut to just $d$ dimensions.

In [ ]:
# Consider just the first 4 dimensions associated to the singular value decomposition matrices.
d_dimensions = 4

cut_U = U[:,:d_dimensions]
cut_Sigma = Sigma[:d_dimensions]
cut_VT = VT[:d_dimensions,:]
print("U:",cut_U.shape)
print("Sigma:",cut_Sigma.shape)
print("VT:",cut_VT.shape)

In [ ]:
# Analyse the contents of U and VT.
def highlight_abs_max(s,n):
  n_largest = np.argsort(s.abs().values)[-n:]
  return ['background-color: yellow' if i in n_largest else '' for i,v in enumerate(s)]
pd.DataFrame(cut_U).style.apply(lambda x: highlight_abs_max(x, 3))
pd.DataFrame(cut_VT.T).style.apply(lambda x: highlight_abs_max(x, 5))

## Principal Component Analysis

For a square matrix $\mathbf{A} \in \mathbb{R}^{n \times n}$ that satisfies the spectral theorem, performing an eigendecomposition consists of finding matrices $\mathbf{\Sigma}, \mathbf{U} \in \mathbb{R}^{n \times n}$ such that
$$\mathbf{A} = \begin{pmatrix}
  \mathbf{u}_1 & \dots & \mathbf{u}_n \\
\end{pmatrix}\begin{pmatrix}
  \lambda_1 & 0 & \dots & 0 \\
  0 & \lambda_2 & \dots & 0 \\
  \vdots & \vdots & \ddots & \vdots \\
  0 & 0 & \dots & \lambda_n \\
\end{pmatrix}\begin{pmatrix}
  \mathbf{u}_1^T \\
  \vdots \\
  \mathbf{u}_n^T \\
\end{pmatrix} = \mathbf{U\Sigma U}^T$$

In [ ]:
# Load and analyse the IRIS Dataset.
iris = datasets.load_iris()
x = iris.data
y = iris.target
labels = iris.target_names[y]

# Print some information about the dataset.
print("X shape",x.shape)
print("Y shape",y.shape)

# Look at some contents of the dataset.
print(x[:10]) # Print the numerical features of the first 10 elements.
print(y) # Print an integer corresponding to the label of each element.
print("Labels", labels) # Print the actual labels of the elements in the dataset.

Principal component analysis is often performed on just $k = 2$ components for visualization purposes.

In [ ]:
# Instantiate a PCA object to perform principal component analysis using 2 components.
n_components = 2
pca = PCA(n_components)
principalComponents = pca.fit_transform(x)
principalDf = pd.DataFrame(data = principalComponents, columns = ['PC1', 'PC2']) # Store information from the principal components in a dataframe.

# Analyse the proportion of data variance explained by the two components.
print("Explained variance ratio per component:",pca.explained_variance_ratio_)
principalDf["Label"] = y # Add the class labels.
principalDf # Take a look at the contents of the dataframe.

In [ ]:
# Plot the results obtained by the principal component analysis algorithm.
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1)
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('PCA - Two Components', fontsize = 20)
targets = range(3)
colors = ['r', 'g', 'b']
for target, color in zip(targets,colors):
  indicesToKeep = principalDf['Label'] == target
  ax.scatter(principalDf.loc[indicesToKeep, 'PC1'],
             principalDf.loc[indicesToKeep, 'PC2'],
             c = color,
             s = 50)
ax.legend(targets)
ax.grid()

Most of the time, performing principal component analysis shows that the first principal component alone explains almost all the data variance but the data has a relationship with the second principal component as well.

In [ ]:
# Consider an example where feature a is randomly generated and features b and c are proportional to it.
num_samples = 100
a = np.random.rand(num_samples)
b = a / 2
c = a / 3
x = np.stack([a, b, c]).T

# Instantiate a PCA object to perform principal component analysis using 2 components.
n_components = 2
pca = PCA(n_components)
principalComponents = pca.fit_transform(x)
principalDf = pd.DataFrame(data = principalComponents, columns = ['PC1', 'PC2'])

# Analyse the proportion of data variance explained by the two components.
print("Explained variance ratio per component:",pca.explained_variance_ratio_)
principalDf = principalDf.round(10)
# principalDf["Label"] = y

# Plot the results obtained by the principal component analysis algorithm.
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1)
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('PCA - Two Components', fontsize = 20)
ax.scatter(principalDf.loc[:, 'PC1'],
           principalDf.loc[:, 'PC2'],
           s = 50)
ax.grid() # The plot reveals that the second principal component actually does not explain any of the variance.

One of the main reasons why principal component analysis might give misleading results lies in the fact that the algorithm is affected by scale.

For this reason, it is necessary to first standardize the dataset by ensuring that it has zero mean and unit variance.

In [ ]:
# Perform principal component analysis on the IRIS Dataset, but scale the features first.
iris = datasets.load_iris()
x = iris.data
y = iris.target
labels = iris.target_names[y]
x = StandardScaler().fit_transform(x)

n_components = 2
pca = PCA(n_components)
principalComponents = pca.fit_transform(x)
principalDf = pd.DataFrame(data = principalComponents, columns = ['PC1', 'PC2'])
principalDf["Label"] = y

# Analyse the proportion of data variance explained by the two components.
print("Explained variance ratio per component:",pca.explained_variance_ratio_)

# Plot the results obtained by the principal component analysis algorithm.
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1)
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('PCA - Two Components', fontsize = 20)
targets = range(3)
colors = ['r', 'g', 'b']
for target, color in zip(targets,colors):
  indicesToKeep = principalDf['Label'] == target
  ax.scatter(principalDf.loc[indicesToKeep, 'PC1']
             ,principalDf.loc[indicesToKeep, 'PC2']
             ,c = color
             ,s = 50)
ax.legend(targets)
ax.grid()

Generally speaking, the number of principal components can be determined manually through the elbow method or automatically by taking the number of features that cumulatively explains a chosen proportion of the total variance.